In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras

# fasion MNIST 데이터 다운로드
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

In [2]:
import numpy as np
def filter_binary(x, y, neg_cls, pos_cls):
    cond = (y == neg_cls) | (y == pos_cls)
    x_bin = x[cond]
    y_bin = np.where(y[cond] == neg_cls, 0, 1)
    return x_bin, y_bin

In [3]:
x_train_binary, y_train_binary = filter_binary(x_train, y_train, 2, 6)
x_test_binary, y_test_binary = filter_binary(x_test, y_test, 2, 6)

In [ ]:
# 이진 분류
model = keras.models.Sequential(
    [keras.layers.Rescaling(1/255),
     keras.layers.Flatten(),
     keras.layers.Dense(16, activation='relu'),    # vanishing gradient 방지하는 activation function
     keras.layers.Dense(1, activation='sigmoid')]) # output layer, 0과 1로 분류하는 sigmoid 함수

model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.001), 
              loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train_binary, y_train_binary, epochs=5, batch_size=32)

Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6715 - loss: 0.6417
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7299 - loss: 0.5965
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7495 - loss: 0.5678
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7612 - loss: 0.5474
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7710 - loss: 0.5318


In [ ]:
# 다항 분류
model = keras.models.Sequential([
    keras.layers.Rescaling(1/255),
    keras.layers.Flatten(),
    keras.layers.Dense(16, activation='relu'),   # 은닉층
    keras.layers.Dense(10, activation='softmax') # 출력층, 10개의 클래스
    # softmax: 여러 개의 입력을 받아 같은 개수를 출력(n개 input -> 10개 output)
    #          -> 모든 출력의 합 = 1, 출력 범위 0 ~ 1
    #          -> 출력 클래스가 3개 이상일 때 사용
])
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.001), 
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=32)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.3088 - loss: 1.9608
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.5462 - loss: 1.4002
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.6625 - loss: 1.0587
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.6958 - loss: 0.9189
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.7171 - loss: 0.8383


In [ ]:
# 예측 확률 계산
y_prob = model.predict(x_test)
y_prob.shape

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


(10000, 10)

In [ ]:
# softmax 함수
keras.activations.softmax(np.array([-1, 0.5, 2.0]))

tensor([0.0391, 0.1753, 0.7856], dtype=torch.float64)

In [12]:
# sigmoid는 softmax의 특수한 경우(이진 분류)
x = 1.0
y_sig = keras.activations.sigmoid(x)
y_soft = keras.activations.softmax(np.array([0.0, 1.0]))
print(y_sig, y_soft)

tensor(0.7311) tensor([0.2689, 0.7311], dtype=torch.float64)


In [13]:
# softmax가 없는 모형
# softmax의 input 값: logit
# -> logit이 높으면 softmax의 출력인 확률도 높음
# 어차피 logit이 가장 높은 클래스를 선택할 거라면, 확률 계산을 생략하고 logit을 바로 이용해서 학습
# -> 연산량 줄일 수 있음
model = keras.models.Sequential([
    keras.layers.Rescaling(1/255),
    keras.layers.Flatten(),
    keras.layers.Dense(16, activation='relu'),   # 은닉층
    keras.layers.Dense(10) # 출력층, softmax 없음
])
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.001), 
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=32)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.4698 - loss: 1.6698
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 62s 33ms/step - accuracy: 0.6885 - loss: 1.0455
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.7241 - loss: 0.8591
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7490 - loss: 0.7681
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.7684 - loss: 0.7097


In [ ]:
# logit 계산
logits = model.predict(x_test)
print(logits.shape)  # 10000, 10
print(logits[0]) # 0번째 샘플의 로짓
print(keras.activations.softmax(logits[0])) # 0번째 샘플의 확률

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
(10000, 10)
[-3.242375  -3.5225432 -1.2200458 -3.2370453 -1.3045865  3.3844337
 -3.1064038  2.7753043  1.1117826  3.5443757]
tensor([4.6556e-04, 3.5181e-04, 3.5178e-03, 4.6805e-04, 3.2326e-03, 3.5153e-01,
        5.3337e-04, 1.9117e-01, 3.6222e-02, 4.1250e-01])


In [ ]:
# 정칙화(regularization): 과적합을 막기 위한 방법들

# - Weight decay: 파라미터를 업데이트할 때마다 줄임
#                 -> 손실함수에 L2 norm을 추가하고, SGD로 파라미터를 업데이트하는 것과 동일
#                 -> L1 Norm: MAE, L2 Norm: MSE
# - Early Stopping: 과적합 방지를 위해 validation loss가 줄어들지 않으면 일찍 멈춤
# - Dropout: 일부 파라미터를 훈련에서 배제
# - Batch/Layer Normalization: 레이어의 출력값 정규화
# - Label Smoothing: 모형 출력의 목표값을 줄임

In [15]:
# Early Stopping을 이용한 훈련
model = keras.models.Sequential([keras.layers.Rescaling(1/255), 
                                 keras.layers.Flatten(), 
                                 keras.layers.Dense(10)])

model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.001), 
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
              metrics=['accuracy'])

model.fit(x_train, y_train, batch_size=32, 
          epochs=50, validation_split=0.2, # 에포크마다 20%의 데이터로 검증
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=1)]) 
          # monitor: 검증 데이터의 손실을 기준으로 멈춤 여부를 결정
          # patience: 몇 에포크 동안 개선이 없으면 멈출지 지정

# -> 성능이 일시적으로 하락해도 정지함 -> 자동 저장 기능 활용

Epoch 1/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.5238 - loss: 1.5791 - val_accuracy: 0.6661 - val_loss: 1.1883
Epoch 2/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.6727 - loss: 1.0716 - val_accuracy: 0.6967 - val_loss: 0.9657
Epoch 3/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.7034 - loss: 0.9239 - val_accuracy: 0.7214 - val_loss: 0.8665
Epoch 4/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7266 - loss: 0.8464 - val_accuracy: 0.7395 - val_loss: 0.8076
Epoch 5/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7429 - loss: 0.7960 - val_accuracy: 0.7512 - val_loss: 0.7662
Epoch 6/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7542 - loss: 0.7595 - val_accuracy: 0.7624 - val_loss: 0.7347
Epoch 7/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7642 - loss: 0.7312 - val_accuracy: 0.7707 - val_loss: 0.7104
Epoch 8/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7714 - loss: 0.7084 - 

In [ ]:
# 자동 저장
# ModelCheckpoint 콜백을 이용해서 검증 성능이 가장 좋은 모델의 파라미터 자동 저장
model.fit(x_train, y_train, epochs=50, batch_size=32, validation_split=0.2,
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3),
                     keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)])

# 저장된 파라미터 불러오기
best_model = keras.models.load_model('best_model.h5')

Epoch 1/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8328 - loss: 0.5042

1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.8329 - loss: 0.5023 - val_accuracy: 0.8282 - val_loss: 0.5048
Epoch 2/50
1493/1500 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8344 - loss: 0.4953

1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.8334 - loss: 0.5009 - val_accuracy: 0.8292 - val_loss: 0.5039
Epoch 3/50
 552/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8349 - loss: 0.4915

In [ ]:
# Dropout
# 학습 중 입력층과 은닉층의 일부 노드(무작위)의 출력값을 0으로 대체
# 예측할 때는 dropout 대신 출력 비율로 조절(학습 시: 50% dropout -> 예측 시: 출력 크기 50%로 축소)
model = keras.models.Sequential([keras.layers.Rescaling(1/255),
                                 keras.layers.Flatten(),
                                 keras.layers.Dense(128, activation='relu'),
                                 keras.layers.Dropout(0.5), # 드롭아웃 비율 50
                                 keras.layers.Dense(10)])

model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.001), 
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
              metrics=['accuracy'])

model.fit(x_train, y_train, epochs=50, batch_size=32, validation_split=0.2,
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3),
                     keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)])

In [ ]:
# Normalization: 값의 크기를 일정하게 만들어주는 것
# Standardization: (x - mu)/sigma <- 평균 = 0, 표준편차 = 1
# e.g. 이미지 normalization: 0 ~ 255 범위의 픽셀 값을 -1 ~ 1 또는 0 ~ 1 범위로 normalize

In [ ]:
# Batch Normalization: batch 단위로 레이어의 입력을 normalize
# 효과:
# 1. 신경망의 출력을 일정 범위로 제한하여 포화 억제
# 2. 높은 학습률에도 안정적으로 학습
# 3. 배치에 따라 평균과 표준편차가 달라지므로 출력값에 일정한 오차 발생 <- Dropout과 비슷한 효과
model = keras.models.Sequential([keras.layers.Rescaling(1/255),
                                 keras.layers.Flatten(),
                                 keras.layers.Dense(128, activation='relu'),
                                 keras.layers.BatchNormalization(), # 배치 정규화
                                 keras.layers.Dense(10)])

model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.001), 
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
              metrics=['accuracy'])

model.fit(x_train, y_train, epochs=50, batch_size=32, validation_split=0.2,
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=3),
                     keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)])

In [ ]:
# one-hot encoding
# 범주형 변수를 0과 1로 표현
# e.g. 신문기사의 주제 분류: [정치, 사회, 경제, 문화] <- 사회 기사:[0, 1, 0, 0]

In [ ]:
# label smoothing
# 과적합 방지를 위해 one-hot encoding에서 1을 조금 줄이고 0을 조금 키움
# e.g. [1, 0, 0, 0] -> [0.85, 0.05, 0.05, 0.05]

keras.losses.BinaryCrossentropy(label_smoothing=0.15) # 이항 분류
keras.losses.CategoricalCrossentropy(label_smoothing=0.15) # 다항 분류